##### STA 220 Data & Web Technologies for Data Analysis

# Lecture 11 - 02/10/26, Selenium

### Announcements
- Second homework assignment will be uploaded soon (Scraping with XPath/BS4)
- This week's discussion topic: Error handling

### Today's topics
 - Selenium Browser

### Ressources
 - [WhereTheISS](wheretheiss.at)
 - [AstroViewer](https://www.astroviewer.net/iss/en/observation.php)

## ISS and Satellite Data

Let's have a look on this great website:
https://wheretheiss.at/

It even provides a documented API! Find the documentation [here](https://wheretheiss.at/w/developer)!

#### Introduction

In [1]:
import requests
import requests_cache
import pandas as pd

headers = {
    'User - agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:142.0) Gecko/20100101 Firefox/142.0',
}

session = requests_cache.CachedSession('../output/ISS3')

The documentation says it contains a lot of satellites plus the ISS.

So, let's first check all available satellites!

In [2]:
url = 'https://api.wheretheiss.at/v1/satellites'

response = session.get(url, headers = headers)
response.raise_for_status()
print(response.headers)

{'Access-Control-Allow-Origin': '*', 'Cache-Control': 'max-age=0, no-cache', 'Connection': 'Keep-Alive', 'Content-Length': '27', 'Content-Type': 'application/json', 'Date': 'Thu, 05 Feb 2026 22:52:44 GMT', 'Keep-Alive': 'timeout=15, max=100', 'Server': 'Apache/2.2.22 (Ubuntu)', 'X-Apache-Time': 'D=20646', 'X-Powered-By': 'PHP/5.3.10-1ubuntu3.26', 'X-Rate-Limit-Interval': '5 minutes', 'X-Rate-Limit-Limit': '350', 'X-Rate-Limit-Remaining': '349'}


In [ ]:
pd.DataFrame(list(response.headers.items()))

We can get the rate limits from the headers!

Now, lets check out all satellites:

In [ ]:
all_satelites = print(response.json())

Only the ISS :(

In [ ]:
iss_id = str(response.json()[0]['id'])

Get the location of the ISS:

In [ ]:
import requests

url = 'https://api.wheretheiss.at/v1/satellites/' + iss_id

response = session.get(url, headers = headers)
response.raise_for_status()
print(response.headers)

In [ ]:
lim_remaining = response.headers["X-Rate-Limit-Remaining"]
print(f'Remaining requests within the next five minutes: {lim_remaining}')

In [ ]:
print(response) # shows the status code, not the content itself

Note: this is a cached response!

In [ ]:
data = response.json()
print(data) # here we go!

In [ ]:
import pandas as pd

tbl = pd.DataFrame(data, index=[0])

In [ ]:
tbl.head() # looks good

<p>Find the developer rate limits <a href="https://wheretheiss.at/w/developer">here</a>!</p>

<div class="alert alert-danger">
    We MUST NOT send more then one request per second!
</div>


#### Accessing the location

Let's access the ISS' location for a specific timestamp:

In [3]:
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
import warnings
import time

In [4]:
def get_positions(timestamps):
    response = session.get('https://api.wheretheiss.at/v1/satellites/' + iss_id + '/positions?timestamps=' + str(timestamps))
    response.raise_for_status()
    lim_remaining = response.headers["X-Rate-Limit-Remaining"]
    if int(lim_remaining) < 10: # only slow down if we have indeed 
        warnings.warn("Too many requests!")
    time.sleep(1)
    return(response.json())

def is_inlighted(data):
    status = data["visibility"]
    return(status == "daylight")

#### Insertion: timedate formats
Dealing with different timedate formats

In [5]:
tz = 'America/Los_Angeles'
def dt2epoch(date):
    tme = np.datetime64(date)
    return(tme.astype(np.int64))

print(dt2epoch("now"))
print(dt2epoch("2025-02-05 15:00:00"))
print(dt2epoch("2025-02-05 15:00:00-0800"))

1770761944
1738767600
1738796400


/var/folders/df/79c6nd413h713v6l3znh208w0000gq/T/ipykernel_25108/3763540327.py:3: UserWarning: no explicit representation of timezones available for np.datetime64
  tme = np.datetime64(date)


In [6]:
tz = 'America/Los_Angeles'
def epoch2dt(epoch):
    return(pd.to_datetime(epoch, unit = "s", utc=True).tz_convert(tz))

In [7]:
current_time = np.datetime64('now') # np.datetime64('2026-02-05 05:00:00-0800') # np.datetime64('now')

In [8]:
current_time

np.datetime64('2026-02-10T22:19:17')

In [9]:
dt2epoch(current_time)

np.int64(1770761957)

In [10]:
epoch2dt(dt2epoch(current_time))

Timestamp('2026-02-10 14:19:17-0800', tz='America/Los_Angeles')

#### END OF TENTH LECTURE

In [11]:
import numpy as np
import pandas as pd
import requests
import requests_cache

session = requests_cache.CachedSession('../output/ISS3')

#### DUSK/DAWN

We consider twilight to be everything shortly before and after dawn or dusk. (This is a very rough proxy!)

https://www.timeanddate.com/sun/usa/davis

In [12]:
current_time

np.datetime64('2026-02-10T22:19:17')

In [13]:
!pip install astral

In [14]:
from astral import LocationInfo
from astral.sun import sun
import datetime as dt

tz = 'America/Los_Angeles'
timestamp_dt = epoch2dt(current_time)
davis = LocationInfo("Davis", "California", tz, 38.544907, -121.740517)
s = sun(davis.observer, date=timestamp_dt.date(), tzinfo=davis.timezone)
sun_times = pd.Series([s['dawn'], s['dusk']])

In [15]:
sun_times

0   2026-02-10 06:35:34.361979-08:00
1   2026-02-10 18:07:18.645880-08:00
dtype: datetime64[ns, America/Los_Angeles]

We are interested in the times shortly before dawn and shortly after dusk. 
To relax this assumption slightly, we also allow for 15 minutes after dawn and 15 minutes before dusk.
Altogether, we are consider time windows of 90 minutes length.

In [16]:
from astral import LocationInfo
from astral.sun import sun
import datetime as dt
import pandas as pd

tz = 'America/Los_Angeles'
davis = LocationInfo("Davis", "California", tz, 38.544907, -121.740517)

today = pd.Timestamp("02-10-2026") # pd.Timestamp("today")
print('Current time: ' + str(today))

twilight = []

for i in range(4):
    day = today + pd.Timedelta(i, "days")
    s = sun(davis.observer, date=day, tzinfo=davis.timezone)

    twilight += [(s['dawn'] - pd.Timedelta(75, "minutes") + pd.Timedelta(sec, "seconds")).timestamp() for sec in range(0,90*60,10)]
    twilight += [(s['dusk'] - pd.Timedelta(15, "minutes") + pd.Timedelta(sec, "seconds")).timestamp() for sec in range(0,90*60,10)]
    print('Dawn for today + ' + str(i) + " day(s): " + str(s['dawn']))
    print('Dusk for today + ' + str(i) + " day(s): " + str(s['dusk']))

Current time: 2026-02-10 00:00:00
Dawn for today + 0 day(s): 2026-02-10 06:35:34.361979-08:00
Dusk for today + 0 day(s): 2026-02-10 18:07:18.645880-08:00
Dawn for today + 1 day(s): 2026-02-11 06:34:30.466978-08:00
Dusk for today + 1 day(s): 2026-02-11 18:08:22.953305-08:00
Dawn for today + 2 day(s): 2026-02-12 06:33:25.195902-08:00
Dusk for today + 2 day(s): 2026-02-12 18:09:27.091324-08:00
Dawn for today + 3 day(s): 2026-02-13 06:32:18.579461-08:00
Dusk for today + 3 day(s): 2026-02-13 18:10:31.049700-08:00


In [18]:
print(twilight[0:2])
print(len(twilight))

[1770729634.361979, 1770729644.361979]
4320


In [ ]:
twilight

#### GET DATA

In [19]:
import tqdm, requests

url = "https://api.wheretheiss.at/v1/satellites/25544/positions?timestamps="

def get_positions(timestamps):
    response = session.get(url + str(timestamps))
    response.raise_for_status()
    lim_remaining = response.headers["X-Rate-Limit-Remaining"]
    if int(lim_remaining) < 20:
        warnings.warn("Too many requests!")
        time.sleep(1)
    time.sleep(1)
    return(response.json())

In [22]:
tmp = str(twilight[0]) + "," + str(twilight[1])

In [23]:
tmp

'1770729634.361979,1770729644.361979'

In [24]:
get_positions(tmp)

[{'name': 'iss',
  'id': 25544,
  'latitude': 47.068226246755,
  'longitude': -12.100094455267,
  'altitude': 419.28853553015,
  'velocity': 27609.675621125,
  'visibility': 'daylight',
  'footprint': 4503.842076143,
  'timestamp': 1770729634,
  'daynum': 2461082.0559491,
  'solar_lat': -14.236078849728,
  'solar_lon': 343.40553571948,
  'units': 'kilometers'},
 {'name': 'iss',
  'id': 25544,
  'latitude': 46.794594607946,
  'longitude': -11.285395972523,
  'altitude': 419.26519242011,
  'velocity': 27609.54468612,
  'visibility': 'daylight',
  'footprint': 4503.7231986025,
  'timestamp': 1770729644,
  'daynum': 2461082.0560648,
  'solar_lat': -14.236041070355,
  'solar_lon': 343.36386941672,
  'units': 'kilometers'}]

In [25]:
pos = []

for ind in tqdm.tqdm(range(0, len(twilight), 10)):
#    progress = np.round(100*ind/len(epoch_time),1)
    timestamps = twilight[ind:(ind+10)]
    ts_string = ",".join(map(str, timestamps))
    
    pos += get_positions(ts_string)
    # time.sleep(1)

print(pos)

  8%|▊         | 35/432 [00:35<06:44,  1.02s/it]


KeyboardInterrupt: 

##### SAVE / LOAD DATA

In [ ]:
import json

with open('../output/ISS3.json','w+') as file:
    json.dump(pos, file)

In [26]:
import json

with open('../output/ISS3.json','r') as file:
    pos = json.load(file)

##### CONTINUE

In [27]:
len(pos)

4320

In [28]:
data = pd.DataFrame(pos)
data

,name,id,latitude,longitude,altitude,velocity,visibility,footprint,timestamp,daynum,solar_lat,solar_lon,units
0,iss,25544,47.162784,-12.412437,419.215143,27610.124686,daylight,4503.468303,1770729634,2.461082e+06,-14.236079,343.405536,kilometers
1,iss,25544,46.891397,-11.594660,419.189972,27610.003068,daylight,4503.340105,1770729644,2.461082e+06,-14.236041,343.363869,kilometers
2,iss,25544,46.613620,-10.785607,419.164295,27609.878097,daylight,4503.209324,1770729654,2.461082e+06,-14.236003,343.322203,kilometers
3,iss,25544,46.329579,-9.985299,419.138149,27609.749740,daylight,4503.076148,1770729664,2.461082e+06,-14.235966,343.280537,kilometers
4,iss,25544,46.039400,-9.193755,419.111571,27609.617964,daylight,4502.940769,1770729674,2.461082e+06,-14.235928,343.238870,kilometers
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4315,iss,25544,-32.715854,-36.974233,436.736474,27526.971776,eclipsed,4591.649450,1771039481,2.461086e+06,-13.041017,132.365454,kilometers
4316,iss,25544,-32.278111,-36.456572,436.509259,27527.565370,eclipsed,4590.519181,1771039491,2.461086e+06,-13.040977,132.323786,kilometers
4317,iss,25544,-31.837867,-35.944238,436.281018,27528.163668,eclipsed,4589.383463,1771039501,2.461086e+06,-13.040938,132.282119,kilometers
4318,iss,25544,-31.395184,-35.437118,436.051827,27528.766527,eclipsed,4588.242667,1771039511,2.461086e+06,-13.040899,132.240451,kilometers


In [29]:
data = data[data.visibility == "daylight"] # only consider the cases where the ISS is in daylight!
perc = int(len(data)/len(pos)*100)

print(f'The ISS is in daylight in {perc}% of the cases.')

The ISS is in daylight in 63% of the cases.


In [30]:
data.head()

,name,id,latitude,longitude,altitude,velocity,visibility,footprint,timestamp,daynum,solar_lat,solar_lon,units
0,iss,25544,47.162784,-12.412437,419.215143,27610.124686,daylight,4503.468303,1770729634,2.461082e+06,-14.236079,343.405536,kilometers
1,iss,25544,46.891397,-11.594660,419.189972,27610.003068,daylight,4503.340105,1770729644,2.461082e+06,-14.236041,343.363869,kilometers
2,iss,25544,46.613620,-10.785607,419.164295,27609.878097,daylight,4503.209324,1770729654,2.461082e+06,-14.236003,343.322203,kilometers
3,iss,25544,46.329579,-9.985299,419.138149,27609.749740,daylight,4503.076148,1770729664,2.461082e+06,-14.235966,343.280537,kilometers
4,iss,25544,46.039400,-9.193755,419.111571,27609.617964,daylight,4502.940769,1770729674,2.461082e+06,-14.235928,343.238870,kilometers


In [32]:
data = data.drop(['name', 'id', 'velocity', 'visibility', 'daynum', 'solar_lat', 'solar_lon', 'units'], axis = 1).set_index("timestamp")
data

KeyError: "['name', 'id', 'velocity', 'visibility', 'daynum', 'solar_lat', 'solar_lon', 'units'] not found in axis"

In [33]:
data

,latitude,longitude,altitude,footprint
timestamp,,,,
1770729634,47.162784,-12.412437,419.215143,4503.468303
1770729644,46.891397,-11.594660,419.189972,4503.340105
1770729654,46.613620,-10.785607,419.164295,4503.209324
1770729664,46.329579,-9.985299,419.138149,4503.076148
1770729674,46.039400,-9.193755,419.111571,4502.940769
...,...,...,...,...
1771038881,-51.080395,-81.224616,445.879698,4636.851526
1771038891,-50.964349,-80.265267,445.833107,4636.622564
1771038901,-50.839775,-79.311062,445.782153,4636.372147


Great! Now we have all moments where the ISS is visible from __some point on this planet__. But what about Davis?!?

For this, let's calculate the distance between Davis and the projection of the ISS on the earth.

Remember: we don't have to do everything by hand. For most applications, there is a Python package.

In [34]:
!pip install geopy

In [35]:
from geopy.distance import geodesic

davis = (38.544907, -121.740517)

def calc_diff(coords):
    point = (coords["latitude"], coords["longitude"])
    return(geodesic(point, davis).km)

In [36]:
dist = data.apply(calc_diff, axis=1)
data["distance"] = dist

In [37]:
data.head()

,latitude,longitude,altitude,footprint,distance
timestamp,,,,,
1770729634,47.162784,-12.412437,419.215143,4503.468303,8214.384349
1770729644,46.891397,-11.594660,419.189972,4503.340105,8281.384545
1770729654,46.613620,-10.785607,419.164295,4503.209324,8348.415620
1770729664,46.329579,-9.985299,419.138149,4503.076148,8415.476973
1770729674,46.039400,-9.193755,419.111571,4502.940769,8482.567214


In [38]:
near = (dist < data["footprint"])
data["is_near"] = near
data.head()

,latitude,longitude,altitude,footprint,distance,is_near
timestamp,,,,,,
1770729634,47.162784,-12.412437,419.215143,4503.468303,8214.384349,False
1770729644,46.891397,-11.594660,419.189972,4503.340105,8281.384545,False
1770729654,46.613620,-10.785607,419.164295,4503.209324,8348.415620,False
1770729664,46.329579,-9.985299,419.138149,4503.076148,8415.476973,False
1770729674,46.039400,-9.193755,419.111571,4502.940769,8482.567214,False


In [39]:
data.shape

(2760, 6)

In [40]:
tbl = data[data["is_near"]]
len(tbl)

218

In [41]:
perc_near = round(len(tbl)/len(data)*100,1)

In [42]:
print('The ISS is near in ' + str(perc_near) + '% of the cases where it is also visible.')

The ISS is near in 7.9% of the cases where it is also visible.


In [43]:
tbl.head()

,latitude,longitude,altitude,footprint,distance,is_near
timestamp,,,,,,
1770733734,12.193201,-141.870128,418.543682,4500.046937,3535.238256,True
1770733744,12.695104,-141.491696,418.466219,4499.652028,3466.479163,True
1770733754,13.196336,-141.111605,418.391701,4499.272088,3397.733597,True
1770733764,13.696869,-140.729782,418.320109,4498.907035,3329.002357,True
1770733774,14.196673,-140.346153,418.251425,4498.556778,3260.286329,True


In [44]:
amin = np.argmin(tbl[tbl.is_near]["distance"])
minimizer = tbl[tbl.is_near].iloc[amin]
print(minimizer)

latitude       40.736435
longitude    -123.669526
altitude      418.404976
footprint    4499.339776
distance      294.299471
is_near             True
Name: 1770907205, dtype: object


In [45]:
epoch2dt(minimizer.name)

Timestamp('2026-02-12 06:40:05-0800', tz='America/Los_Angeles')

Compare it with: https://www.astroviewer.net/iss/en/observation.php

#### BEARING

In [46]:
tz = 'America/Los_Angeles'
davis = [38.544907, -121.740517]

import math

def calculate_bearing(point):
    """
    Calculates the bearing in degrees between two geolocations.
    Latitudes and longitudes should be provided in decimal degrees.
    """
    lat1_rad = math.radians(davis[0])
    lon1_rad = math.radians(davis[1])
    lat2_rad = math.radians(point[0])
    lon2_rad = math.radians(point[1])

    delta_lon = lon2_rad - lon1_rad

    y = math.sin(delta_lon) * math.cos(lat2_rad)
    x = math.cos(lat1_rad) * math.sin(lat2_rad) - \
        math.sin(lat1_rad) * math.cos(lat2_rad) * math.cos(delta_lon)

    bearing_rad = math.atan2(y, x)
    bearing_deg = math.degrees(bearing_rad)
    
    # Normalize bearing to be within 0-360 degrees
    bearing_deg = (bearing_deg + 360) % 360
    return bearing_deg

def get_cardinal_direction(point):
    """
    Converts a bearing in degrees (0-360) to a cardinal direction.
    """
    directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    # Adjust bearing to align with the center of each 45-degree sector
    # For example, N is centered at 0, NE at 45, E at 90, etc.
    bearing = calculate_bearing(point)
    index = round(bearing / 45) % 8 # use modulo since 359 degree should also be 0 (N)
    return directions[index]

##### Old Example

In [112]:
# Example Usage:
point = [38.8977, -77.0365] 

bearing = calculate_bearing(point)
direction = get_cardinal_direction(point)

print(f"The bearing from is {bearing:.2f} degrees.")
print(f"The cardinal direction is: {direction}")

The bearing from is 75.03 degrees.
The cardinal direction is: E


The example point was the White House. It is indeed eastern of Davis, see:
[Google Maps](https://www.google.com/maps/dir/University+of+California,+Shields+Avenue,+Davis,+Kalifornien/38.8977,-77.0365/@38.2386811,-109.987335,5z/data=!3m1!4b1!4m9!4m8!1m5!1m1!1s0x80ead37f7489fa3f:0xecbfbb24087e8334!2m2!1d-121.7617125!2d38.5382322!1m0!3e4?entry=ttu&g_ep=EgoyMDI1MDkxNy4wIKXMDSoASAFQAw%3D%3D)

##### NEW EXAMPLE

In [47]:
point = [37.7576928,-122.4787994]

bearing = calculate_bearing(point)
direction = get_cardinal_direction(point)

print(f"The bearing from is {bearing:.2f} degrees.")
print(f"The cardinal direction is: {direction}")

The bearing from is 216.64 degrees.
The cardinal direction is: SW


The example point was the San Fracisco. It is indeed south-western of Davis, see:
[Google Maps](https://www.google.com/maps/dir/Davis,+California/San+Francisco,+California/@38.0148857,-122.2645703,8.99z/data=!4m14!4m13!1m5!1m1!1s0x808529999495543f:0xc3013f1b6ee28fff!2m2!1d-121.7405167!2d38.5449065!1m5!1m1!1s0x80859a6d00690021:0x4a501367f076adff!2m2!1d-122.4194155!2d37.7749295!3e3?entry=ttu&g_ep=EgoyMDI2MDIwOC4wIKXMDSoASAFQAw%3D%3D)

In [48]:
point = list(minimizer[["latitude", "longitude"]])
print(calculate_bearing(point))
print(get_cardinal_direction(point))

326.48127937513965
NW


Compare it with: https://www.astroviewer.net/iss/en/observation.php!

Waaaaaiiiit. Couldn't we just scrape this site?

In [49]:
import requests
import requests_cache
import pandas as pd
import lxml.html as lx

In [50]:
url = 'https://www.astroviewer.net/iss/en/observation.php'
headers = {
    'User - agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:142.0) Gecko/20100101 Firefox/142.0',
}

In [51]:
response = requests.get(url)
response.raise_for_status()

In [52]:
html = lx.fromstring(response.text)

In [53]:
table = html.xpath('//table[contains(@class, "passDetails")]')[0]

In [54]:
rows = table.xpath('//tbody/tr')

In [55]:
cells = table.xpath('//tbody/tr/td')

In [56]:
[c.text for c in cells]

['...',
 '...',
 '...',
 '...',
 '...',
 '...',
 '...',
 '...',
 '...',
 '...',
 '...',
 None]

In [ ]:
[c.text_content() for c in rows[0].xpath('//td')]

It conains only '...'. Have a look at the HTML content!

And how to choose Davis in the first place?

## --> SELENIUM BROWSER

# Selenium WebDriver

## Preparations

Before diving into Selenium’s features, let’s **install Chrome**, **configure ChromeDriver**, and install the Python **selenium** package.

Alternatively, you may also use the geckodriver for Firefox instead. See [here](https://www.selenium.dev/documentation/webdriver/browsers/firefox/) for more details about using Firefox through the geckodriver.

### Install the Selenium Library

In [57]:
!pip install selenium

### Install the Browser Driver

There are two ways to set up a browser driver for Chrome:

1. **Manual Installation**  
   - Check your local **Chrome** version by typing `chrome://version` in Chrome’s address bar or via “Help → About Google Chrome.”  
   - Download the matching **ChromeDriver** from  
     <https://chromedriver.storage.googleapis.com/index.html>  
   - Either add the `chromedriver.exe` to your system’s PATH (e.g., drop it into Python’s `Scripts/` folder) or specify the absolute path directly in your code.

2. **Automatic Installation**  
   - Use a 3rd-party library such as **webdriver_manager** to install the appropriate driver automatically:

In [58]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager

ChromeDriverManager().install() # detects your Chrome version, downloads the matching driver, and places it in your local cache.

'/Users/ndamann/.wdm/drivers/chromedriver/mac64/144.0.7559.133/chromedriver-mac-arm64/chromedriver'

With this setup in place, we can start using Selenium.

## Basic Usage

This section covers **initializing the browser**, visiting pages, setting the **browser window size**, **refreshing**, **forward/back** navigation, etc.

### Initialize a Browser Object

In [ ]:
import time

In [60]:
# Option A: Direct initialization if ChromeDriver is in PATH
driver = webdriver.Chrome()

# Option B: Specify the absolute path to chromedriver
# path = r'C:\path\to\chromedriver.exe' for Windows
# driver = webdriver.Chrome(path)

time.sleep(2)

driver.close()  # Closes the browser

### Access a Page

In [61]:
import time

url = 'https://statistics.ucdavis.edu/'

with webdriver.Chrome() as driver:
    driver.get(url)
    time.sleep(3)
# driver.close() automatically closes the window afterwards

### Headless Browser

Having a browser doing things might be distracting. Let's use the headless mode!

In [62]:
option = webdriver.ChromeOptions()
option.add_argument("headless") # no browser window visible

with webdriver.Chrome(options=option) as driver:
    driver.get(url)
    time.sleep(3)

### Screenshot

While using the headless mode may be useful in practice, you may take a screenshot sometimes, e.g., if an error occurs:

In [63]:
with webdriver.Chrome(options=option) as driver:
    driver.get(url)
    driver.get_screenshot_as_file('../output/screenshot_ucd.png')

![Screenshot](../output/screenshot_ucd.png)

Well, that's only one quarter of the page. Seems like the browser windows is quite small, eh?

### Window Size

In [64]:
with webdriver.Chrome() as driver:
    driver.maximize_window()          # Fullscreen
    driver.get(url)
#    driver.get_screenshot_as_file('../output/screenshot_ucd_max.png')
    time.sleep(2)

    driver.set_window_size(500, 500)  # 500 x 500
    time.sleep(2)

    driver.set_window_size(1000, 800) # 1000 x 800
    driver.get_screenshot_as_file('../output/screenshot_ucd_large.png')
    time.sleep(2)

![Screenshot](../output/screenshot_ucd_large.png)

### Page refresh

In [66]:
url_dynamic = 'https://the-internet.herokuapp.com/dynamic_content'

with webdriver.Chrome() as driver:
    driver.maximize_window()          # Fullscreen
    driver.get(url_dynamic)
    time.sleep(5)
    driver.refresh()
    print('Page refreshed.')
    time.sleep(2)

Page refreshed.


### Forward/Back Navigation

In [67]:
with webdriver.Chrome() as driver:
    driver.get(url_dynamic)
    time.sleep(2)
    driver.get(url)
    time.sleep(2)
    driver.back() # go back to the-internet
    time.sleep(2)
    driver.forward() # go forward to ucd
    time.sleep(2)

## Page Properties
Once Selenium opens a page, you can retrieve basic info:

In [68]:
with webdriver.Chrome() as driver:
    driver.get(url)
    print(driver.title)       # page title
    print(driver.current_url) # current URL
    print(driver.name)        # browser name
    html = driver.page_source # raw HTML source

UC Davis Statistics
https://statistics.ucdavis.edu/
chrome


In [69]:
html[:100]

'<html lang="en" dir="ltr" prefix="og: https://ogp.me/ns#" class=" js" style="--page-width: 1200px; -'

In [70]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(html)
links = soup.find_all('link')

In [71]:
for link in links:
    print(link.get('href'))

https://statistics.ucdavis.edu/
https://statistics.ucdavis.edu/
https://cdn.skypack.dev/pin/lit@v2.0.0-rc.3-RFrIXWBysJfo8GpKQ4Gc/mode=imports,min/optimized/lit.js
/profiles/sitefarm/themes/sitefarm_one/dist/primary-nav.js
https://campusfont.ucdavis.edu/proxima-nova/proximanova_bold_macroman/proximanova-bold-webfont.woff2
https://campusfont.ucdavis.edu/proxima-nova/proximanova_regular_macroman/proximanova-regular-webfont.woff2
https://campusfont.ucdavis.edu/proxima-nova/proximanova_extrabold_macroman/proximanova-extrabold-webfont.woff2
https://use.fontawesome.com/releases/v6.7.2/webfonts/fa-solid-900.woff2
/sites/g/files/dgvnsk5166/files/stats%20chart%20favicon_0.png
/sites/g/files/dgvnsk5166/files/css/css_-9UtQ61-omgDChGy_Tk7vD_nqVrNQ5YwZddpuDPyFHs.css?delta=0&language=en&theme=sitefarm_one&include=eJxljs0OgzAMg18I0dOeJyqtYRFNgpoCYk-_vwtjF0v-bMlOMzI3qzdayjqx0hLTTKwZ2mgoluZwNl2yiqBWJRZ-oHNuGGMVMkUYol_IaNriDje5BAL3OMF_qVq-9M6mc0scC8nrcaTCOnv4R3274712eIN8L621EGRADhW-mDpvIG9Hee1vjN3DR3uxvB

## Page Elements

When using Selenium, a **key** step is to locate elements for input, clicking, etc. Below are common methods.

### Locating Page Elements

The syntax will always be 

```python
driver.find_element(By.X, "your_element_id")
```

where X is either an ID, Tag Name, etc

#### Locate by ID

In [72]:
from selenium.webdriver.common.by import By
url_ab = 'https://the-internet.herokuapp.com/abtest'

with webdriver.Chrome() as driver:
    driver.get(url_ab)
    content = driver.find_element(By.ID, 'content')
    time.sleep(3)
    text = content.text

In [73]:
print(text)

A/B Test Variation 1
Also known as split testing. This is a way in which businesses are able to simultaneously test and learn different versions of a page to see which text and/or functionality works best towards a desired outcome (e.g. a user action such as a click-through).


In [74]:
url_test = 'https://automationintesting.com/selenium/testpage/'

with webdriver.Chrome() as driver:
    driver.get(url_test)
    driver.maximize_window()          # Fullscreen
    element = driver.find_element(By.ID, 'firstname')
    driver.execute_script("arguments[0].scrollIntoView();", element) # scroll until we can see the element
    time.sleep(2)
    element.send_keys('Aggies')
    time.sleep(5)

#### Locate by Name

In [75]:
url_test = 'https://automationintesting.com/selenium/testpage/'

with webdriver.Chrome() as driver:
    driver.get(url_test)
    driver.maximize_window()          # Fullscreen
    element = driver.find_element(By.NAME, 'colour')
    driver.execute_script("window.scrollBy(0, 500);")  # scroll down by 500 pixels
    time.sleep(2)
    element.click()
    time.sleep(2)

#### Locate by Class Name

In [76]:
url_test = 'https://automationintesting.com/selenium/testpage/'

with webdriver.Chrome() as driver:
    driver.get(url_test)
    element = driver.find_element(By.CLASS_NAME, 'info-title')
    title = element.text
    time.sleep(2)

print(title)

SELENIUM TEST PAGE


#### Locate by Tag Name

```python
browser.find_element(By.ID, 'name')
browser.find_element(By.NAME, 'name')
browser.find_element(By.CLASS_NAME, 'name')
browser.find_element(By.TAG_NAME, 'name')
browser.find_element(By.LINK_TEXT, 'name')
browser.find_element(By.PARTIAL_LINK_TEXT, 'name')
browser.find_element(By.XPATH, '//*[@id="name"]')
browser.find_element(By.CSS_SELECTOR, '#name')
```

Note that finding elements by using commands like
```browser.find_element_by_css_selector('#kw')```
are deprecated. It is highly recommended to use the `By.CSS_SELECTOR` instead.

In [ ]:
url_test = 'https://automationintesting.com/selenium/testpage/'

with webdriver.Chrome() as driver:
    driver.get(url_test)
    element = driver.find_element_by_name('info-title') # throws an error message
    title = element.text
    time.sleep(2)

print(title)

### Multiple Elements

If there are multiple matches, use `find_elements_...()` to get a **list** of matching elements.

## 4. Getting Element Attributes

### `get_attribute()`

For example, retrieving the `src` of an `<img>` element:

In [77]:
url = 'https://statistics.ucdavis.edu/'

with webdriver.Chrome() as driver:
    driver.get(url)
    time.sleep(1)
    element = driver.find_element(By.XPATH, '//*[@id="block-hbwelcometotheucdavisdepartmentofstatistics"]/div/img')
    img_src = element.get_attribute('src')
    time.sleep(3)

print(img_src)

https://statistics.ucdavis.edu/sites/g/files/dgvnsk5166/files/styles/sf_title_banner/public/media/images/MSB%20Sept%202021.jpg?h=aba4661c&itok=yKUUi6__


### Getting Text

In [78]:
url = 'https://statistics.ucdavis.edu/'

with webdriver.Chrome() as driver:
    driver.get(url)
    time.sleep(1)
    elements = driver.find_elements(By.XPATH, '//a')
    for el in elements:
        if el.text:
            print(el.text + ": " + el.get_attribute('href'))
    time.sleep(3)

Skip to main content: https://statistics.ucdavis.edu/#main-content
Home: https://statistics.ucdavis.edu/
About: https://statistics.ucdavis.edu/about-us
Courses: https://statistics.ucdavis.edu/courses
Seminars/Events: https://statistics.ucdavis.edu/seminars
Undergraduate: https://statistics.ucdavis.edu/undergrad
Graduate: https://statistics.ucdavis.edu/grad
Stat Lab: https://statistics.ucdavis.edu/stat-lab
Graduate Programs: https://statistics.ucdavis.edu/grad
Learn More: https://statistics.ucdavis.edu/grad/phd
How to Apply: https://statistics.ucdavis.edu/grad/admissions
Learn More: https://statistics.ucdavis.edu/grad/ms
How to Apply: https://statistics.ucdavis.edu/grad/admissions
Undergraduate Programs: https://statistics.ucdavis.edu/undergrad
Learn More: https://statistics.ucdavis.edu/undergrad/data-science/bs-foundations-track
Apply to UC Davis: https://www.ucdavis.edu/admissions/undergraduate
Learn More: https://statistics.ucdavis.edu/undergrad/major-programs#stamajor
Apply to UC Da

#### Other Attributes

In [80]:
url = 'https://statistics.ucdavis.edu/'

with webdriver.Chrome() as driver:
    driver.get(url)
    time.sleep(1)
    logo = driver.find_element(By.XPATH, '//*[@id="block-hbwelcometotheucdavisdepartmentofstatistics"]/div/img')

    print(logo.id)
    print(logo.location)
    print(logo.tag_name)
    print(logo.size)

    time.sleep(3)

f.1BB2E37E076BBBF6D79567D4680AE780.d.3014104F12EB432F8DD6B8F19C565D91.e.9
{'x': 48, 'y': 265}
img
{'height': 254, 'width': 1104}


### Page Interaction

We have already seen some interactions: 
- scrolling
- button clicks
- writing text

In [81]:
url_test = 'https://automationintesting.com/selenium/testpage/'

driver = webdriver.Chrome()
driver.get(url_test)

In [82]:
element = driver.find_element(By.ID, 'firstname')
driver.execute_script("arguments[0].scrollIntoView();", element)

In [83]:
element.send_keys('Aggies') # write text

In [84]:
element.clear() # clear field

In [85]:
element.send_keys('NewAggies')

In [86]:
driver.find_element(By.ID, 'submitbutton').click() # press button

In [87]:
driver.find_element(By.ID, 'submitbutton').submit() # press enter

In [88]:
driver.find_element(By.ID, 'gender').click()
time.sleep(0.3)
driver.find_element(By.XPATH, "//option[@value='my_business']").click()
time.sleep(0.3)
element = driver.find_element(By.ID, 'firstname').send_keys('Aggies')

In [90]:
continents = driver.find_element(By.ID, "continent")

# Examples of what you can do:
print(continents.text)          # Get the visible text
print(continents.is_selected()) # Check if it is currently picked

Asia
Africa
North America
South America
Antarctica
Europe
Australia
False


In [91]:
driver.quit()

## Delayed Waiting

Sometimes elements load dynamically. We have:

1. **`time.sleep(n)`** – forcibly pause n seconds.
2. **Implicit Wait**: `browser.implicitly_wait(10)`
3. **Explicit Wait**: with `WebDriverWait`.

In [92]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome()
driver.get('https://www.astroviewer.net/iss/en/observation.php')

time.sleep(2)
input_field = driver.find_element(By.ID, "locSearch") 
input_field.send_keys("Davis, CA")
time.sleep(1)
input_field.send_keys(Keys.ENTER)
time.sleep(5)
driver.quit()

In [93]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

browser = webdriver.Chrome()
browser.get('https://www.astroviewer.net/iss/en/observation.php')

results_div = browser.find_element(By.ID, "passesHeader")
old_text = results_div.text
print(old_text)

time.sleep(2)
lc_box = browser.find_element(By.ID, 'locSearch')
lc_box.send_keys('Davis, CA')
time.sleep(1)
lc_box.send_keys(Keys.ENTER)

# Wait for the text to change (Custom Lambda)
WebDriverWait(browser, 10).until(
    lambda d: d.find_element(By.ID, "passesHeader").text != old_text
)

print('Website has successfully loaded.')

Visible ISS passes over New York City
Website has successfully loaded.


In [ ]:
driver.quit()

## Concluding Remarks

- **Selenium** is powerful for automating and scraping **dynamic** or **JavaScript-heavy** pages.  
- **Locating elements** can be done via ID, name, class, tag, link text, partial link text, XPath, or CSS.  
- **Mouse** and **keyboard** actions can simulate real user behavior.  
- Combine Selenium with **WebDriverWait** for reliability on sites with asynchronous loading.  
- Don’t forget best practices like **closing** the browser (`browser.quit()`) and being mindful about rate-limiting or server load.

For more advanced examples or a comprehensive PDF, refer to the original blog or advanced Selenium documentation.